<a href="https://colab.research.google.com/github/rlakshmi2k24/Learning_journey_1/blob/main/02_Visit_risk_classification_f.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install xgboost
!pip install catboost
!pip install imbalanced-learn
!pip install shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

df_visit_risk = pd.read_csv("/content/sample_data/healthcare_analytical_dataset.csv")

print(df_visit_risk.shape)
df_visit_risk.head()

(25000, 26)


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,approved_amount,claim_status,payment_days,billing_date,revenue_gap,senior_citizen_flag,long_stay_flag,payment_delay_flag,claim_rejected_flag,approval_ratio
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.00,Rejected,16.0,2025-06-18,23577.37,1,0,0,1,0.0
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,38178.81,Paid,18.0,2025-10-09,0.00,0,0,0,0,1.0
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,5038.97,Paid,NaN,2025-01-20,0.00,0,0,0,0,1.0
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,22813.34,Paid,16.0,2025-12-24,0.00,1,0,0,0,1.0
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,27106.95,Paid,14.0,2025-09-23,0.00,1,0,0,0,1.0


In [3]:
#Visit Count
visit_frequency = (
    df_visit_risk.groupby("patient_id")
      .size()
      .reset_index(name="visit_count")
)

df_visit_risk = df_visit_risk.merge(
    visit_frequency,
    on="patient_id",
    how="left"
)
#Previous Average Stay
patient_avg_los = (
    df_visit_risk.groupby("patient_id")
      ["length_of_stay_hours"]
      .mean()
      .reset_index(name="patient_avg_los")
)

df_visit_risk = df_visit_risk.merge(
    patient_avg_los,
    on="patient_id",
    how="left"
)
#Previous High Risk Ratio
patient_high_risk_rate = (
    df_visit_risk.groupby("patient_id")
      ["risk_score"]
      .apply(
          lambda x:
          (x=="High").mean()
      )
      .reset_index(
          name="patient_high_risk_rate"
      )
)

df_visit_risk = df_visit_risk.merge(
    patient_high_risk_rate,
    on="patient_id",
    how="left"
)
#Department Historical Risk
dept_high_risk_rate = (
    df_visit_risk.groupby("department")
      ["risk_score"]
      .apply(
          lambda x:
          (x=="High").mean()
      )
      .reset_index(
          name="dept_high_risk_rate"
      )
)

df_visit_risk = df_visit_risk.merge(
    dept_high_risk_rate,
    on="department",
    how="left"
)
#Doctor Historical Risk
doctor_risk_rate = (
    df_visit_risk.groupby("doctor_id")
      ["risk_score"]
      .apply(
          lambda x:
          (x=="High").mean()
      )
      .reset_index(
          name="doctor_high_risk_rate"
      )
)

df_visit_risk = df_visit_risk.merge(
    doctor_risk_rate,
    on="doctor_id",
    how="left"
)
#Age Buckets
df_visit_risk["age_bucket"] = pd.cut(
    df_visit_risk["age"],
    bins=[0,18,40,60,120],
    labels=[
        "Child",
        "Adult",
        "Middle",
        "Senior"
    ]
)
#Chronic Patient Flag
df_visit_risk["high_risk_chronic"] = np.where(
    (df_visit_risk["chronic_flag"]==1)
    &
    (df_visit_risk["age"]>=60),
    1,
    0
)
#Age Risk Score
df_visit_risk["age_risk_score"] = np.select(
    [
        df_visit_risk["age"] < 18,
        df_visit_risk["age"] < 60,
        df_visit_risk["age"] >= 60
    ],
    [1,2,3]
)
#Chronic Disease Severity
df_visit_risk["chronic_age_interaction"] = (
    df_visit_risk["chronic_flag"]
    *
    df_visit_risk["age"]
)

#Department Risk Ranking
dept_rank = (
   df_visit_risk.groupby("department")
      ["risk_score"]
      .apply(lambda x:(x=="High").mean())
      .rank()
)

df_visit_risk["department_risk_rank"] = (
    df_visit_risk["department"]
      .map(dept_rank)
)
#Historical Visit Burden
df_visit_risk["patient_visit_burden"] = (
    np.log1p(df_visit_risk["visit_count"])
)
#Visit Type Risk
visit_risk = (
    df_visit_risk.groupby("visit_type")
      ["risk_score"]
      .apply(lambda x:(x=="High").mean())
)

df_visit_risk["visit_type_risk"] = (
    df_visit_risk["visit_type"]
      .map(visit_risk)
)

#Risk Model Dataset
target = "risk_score"

features = [

    "age",
    "gender",
    #"city",
    #"insurance_provider",
    "chronic_flag",
    #"department",
    "visit_type",
    #"doctor_id",
    "visit_count",
    "patient_avg_los",
    "patient_high_risk_rate",
    "dept_high_risk_rate",
    #"doctor_high_risk_rate",
    "age_bucket",
    "high_risk_chronic",
    "age_risk_score",
    "chronic_age_interaction",
    "department_risk_rank",
    "patient_visit_burden",
    "visit_type_risk"
]

risk_df = (
    df_visit_risk[features + [target]]
      .dropna()
)

In [4]:
from sklearn.preprocessing import LabelEncoder

X = risk_df[features]

y = risk_df[target]

categorical_cols = (
    X.select_dtypes(
        include=[
            "object",
            "category"
        ]
    )
    .columns
)

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

    encoders[col] = le

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(y)

print(
    dict(
        zip(
            target_encoder.classes_,
            range(
                len(
                    target_encoder.classes_
                )
            )
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


/tmp/ipykernel_991/2741453598.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(
/tmp/ipykernel_991/2741453598.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(
/tmp/ipykernel_991/2741453598.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.htm

In [5]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X,
    y_encoded
)

mi_df = pd.DataFrame({
    "Feature": X.columns,
    "Score": mi_scores
})

mi_df.sort_values(
    "Score",
    ascending=False
)

,Feature,Score
6,patient_high_risk_rate,0.118768
1,gender,0.007566
13,patient_visit_burden,0.003673
3,visit_type,0.003208
11,chronic_age_interaction,0.001968
4,visit_count,0.001661
12,department_risk_rank,0.001079
5,patient_avg_los,0.000812
2,chronic_flag,0.000281
0,age,0.000000


In [6]:
selected_features = (
    mi_df
      .head(12)
      ["Feature"]
      .tolist()
)

X = X[selected_features]

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y_encoded,
        test_size=0.20,
        stratify=y_encoded,
        random_state=42
    )
)

In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = (
    smote.fit_resample(
        X_train,
        y_train
    )
)

In [9]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
#Train the Model with RandomForest with the parameters Best Random Forest Parameters:
#{'n_estimators': 300, 'min_samples_split': 5, 'max_depth': None}

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_split=5,
    max_depth=None,
    random_state=42,
    class_weight="balanced"
)

rf.fit(
    X_train_smote,
    y_train_smote
)

RandomForestClassifier(class_weight='balanced', min_samples_split=5,
                       n_estimators=300, random_state=42)

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf_1 = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

rf_grid_1 = {

    "n_estimators":[
        200,
        300,
        500
    ],

    "max_depth":[
        5,
        10,
        15,
        None
    ],

    "min_samples_split":[
        2,
        5,
        10
    ]
}

rf_search_1 = RandomizedSearchCV(
    rf_1,
    rf_grid_1,
    scoring="f1_macro",
    cv=cv,
    n_iter=20,
    n_jobs=-1,
    random_state=42
)

rf_search_1.fit(
    X_train_smote,
    y_train_smote
)

best_rf_1 = rf_search_1.best_estimator_
print("Best Random Forest Parameters:")
print(rf_search_1.best_params_)

KeyboardInterrupt: 

In [14]:
from xgboost import XGBClassifier

xgb_1 = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    random_state=42
)

xgb_grid_1 = {

    "n_estimators":[
        200,
        300,
        500
    ],

    "max_depth":[
        3,
        5,
        7
    ],

    "learning_rate":[
        0.01,
        0.05,
        0.1
    ],

    "subsample":[
        0.8,
        1.0
    ]
}

xgb_search_1 = RandomizedSearchCV(
    xgb_1,
    xgb_grid_1,
    scoring="f1_macro",
    cv=cv,
    n_iter=20,
    n_jobs=-1,
    random_state=42
)

xgb_search_1.fit(
    X_train_smote,
    y_train_smote
)

best_xgb_1 = xgb_search_1.best_estimator_
print("Best xgboost Parameters:")
print(xgb_search_1.best_params_)

Best xgboost Parameters:
{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.1}


In [14]:
from xgboost import XGBClassifier

xgb_1 = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    subsample=0.8,
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    random_state=42
)

xgb_1.fit(
    X_train_smote,
    y_train_smote
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None, num_class=3, ...)

In [12]:
from catboost import CatBoostClassifier

cat_1 = CatBoostClassifier(
    verbose=0,
    random_state=42
)

cat_grid_1 = {

    "depth":[
        4,
        6,
        8
    ],

    "learning_rate":[
        0.01,
        0.05,
        0.1
    ],

    "iterations":[
        200,
        500
    ]
}

cat_search_1 = RandomizedSearchCV(
    cat_1,
    cat_grid_1,
    scoring="f1_macro",
    cv=cv,
    n_iter=15,
    n_jobs=-1,
    random_state=42
)

cat_search_1.fit(
    X_train_smote,
    y_train_smote
)

best_cat_1 = cat_search_1.best_estimator_
print("Best Catboost Parameters:")
print(cat_search_1.best_params_)

NameError: name 'RandomizedSearchCV' is not defined

In [16]:
#Train the model using CatBoost with Parameter Best xgboost Parameters:
#{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.1}

from catboost import CatBoostClassifier

cat_1 = CatBoostClassifier(
    verbose=0,
    subsample=0.8,
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    random_state=42,
    bootstrap_type='Bernoulli'
)

cat_1.fit(
    X_train_smote,
    y_train_smote
)

CatBoostClassifier(bootstrap_type='Bernoulli', learning_rate=0.1, max_depth=7, n_estimators=500, random_state=42, subsample=0.8, verbose=0)

In [18]:
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

models_1 = {

    "Random Forest": rf,

    "XGBoost": xgb_1,

    "CatBoost": cat_1
}

results_1 = []

for name, model_1 in models_1.items():

    pred_1 = model_1.predict(X_test)

    results_1.append({

        "Model": name,

        "Accuracy":
        accuracy_score(
            y_test,
            pred_1
        ),

        "F1 Macro":
        f1_score(
            y_test,
            pred_1,
            average="macro"
        )
    })

results_df_1 = pd.DataFrame(results_1)

results_df_1.sort_values(
    "F1 Macro",
    ascending=False
)

,Model,Accuracy,F1 Macro
0,Random Forest,0.4496,0.409325
2,CatBoost,0.4908,0.399911
1,XGBoost,0.4502,0.387095


In [19]:
df_visit_risk["risk_score"].value_counts()
mi_df.head(20)
# sort the features by score decending
mi_df.sort_values(
    "Score",
    ascending=False
)

,Feature,Score
6,patient_high_risk_rate,0.118768
1,gender,0.007566
13,patient_visit_burden,0.003673
3,visit_type,0.003208
11,chronic_age_interaction,0.001968
4,visit_count,0.001661
12,department_risk_rank,0.001079
5,patient_avg_los,0.000812
2,chronic_flag,0.000281
0,age,0.000000


Analysis
Top Feature MI Score
0.20 - 0.60

best feature:

patient_high_risk_rate = 0..1184

Everything else is almost noise.
What This Means?
The model is basically learning from:
patient_high_risk_rate and almost nothing else.

This is why:
Random Forest = 0.41
XGBoost = 0.40
CatBoost = 0.42
The bottle neck is with the Feature Quality but not the Algorithm

In [20]:
# Verify Risk Score Distribution
print(risk_df["risk_score"].value_counts())

print(
    risk_df["risk_score"].value_counts(normalize=True)
)

risk_score
Low       12470
Medium     7496
High       5034
Name: count, dtype: int64
risk_score
Low       0.49880
Medium    0.29984
High      0.20136
Name: proportion, dtype: float64


Risk score disctribution shows

The target is not perfectly balanced, which is realistic for hospitals.

Most visits should indeed be Low Risk.

The dataset is moderately imbalanced:

Low : Medium : High
50 : 30 : 20

A naïve model predicting only:

Low

would already achieve:

~50% Accuracy

CatBoost accuracy is:

48.2%

which is actually worse than the majority-class baseline.

This strongly suggests that the model is struggling to find meaningful patterns.

In [21]:
#Check the Target Itself
risk_profile = (
    df_visit_risk.groupby("risk_score")
      .agg({
          "age":["mean","median"],
          "length_of_stay_hours":["mean","median"],
          "chronic_flag":"mean",
          "visit_count":"mean"
      })
)

risk_profile

age        length_of_stay_hours         chronic_flag  \
                 mean median                 mean  median         mean   
risk_score                                                               
High        44.781287   45.0            19.758576  18.215     0.494835   
Low         44.692141   45.0            19.151184  18.080     0.503448   
Medium      44.886339   45.0            20.078663  18.520     0.506403   

           visit_count  
                  mean  
risk_score              
High          5.933055  
Low           5.997835  
Medium        5.918090

| Risk   | Avg Age | Median Age | Avg LOS | Median LOS | Chronic % | Avg Visits |
| ------ | ------: | ---------: | ------: | ---------: | --------: | ---------: |
| High   |   44.78 |         45 |   19.76 |      18.21 |    49.48% |       5.93 |
| Low    |   44.69 |         45 |   19.15 |      18.08 |    50.34% |       5.99 |
| Medium |   44.89 |         45 |   20.08 |      18.52 |    50.64% |       5.92 |

The dataset has simlar values across 3 different classes

Age:
High   = 44.78
Low    = 44.69
Medium = 44.89

Difference < 0.2 years
Length of Stay:
High   = 19.76
Low    = 19.15
Medium = 20.08

Difference < 1 hour
Chronic Flag:
High   = 49.48%
Low    = 50.34%
Medium = 50.64%

Difference < 1.2%
Visit Count:
High   = 5.93
Low    = 5.99
Medium = 5.92

Almost identical
Why F1 = 0.42 is Actually Expected

A doctor trying to classify:

Patient A:
Age = 45
LOS = 19
Chronic = Yes

Patient B:
Age = 45
LOS = 20
Chronic = Yes

One is labeled:High

and another:Low

There is no learnable pattern.

Even humans cannot reliably distinguish them.

Therefore:

CatBoost ≈ 0.42
RandomForest ≈ 0.41
XGBoost ≈ 0.40



In [22]:
# copy df_visit_risk to another data frame
risk_df_1 = df_visit_risk.copy()

# Rebuild the Risk Score using Healthcare Business Logic.
def calculate_risk(row):

    score = 0

    if row["age"] >= 65:
        score += 2

    if row["chronic_flag"] == 1:
        score += 2

    if row["length_of_stay_hours"] > 24:
        score += 2

    if row["visit_count"] > 8:
        score += 1

    if score >= 5:
        return "High"

    elif score >= 3:
        return "Medium"

    else:
        return "Low"

risk_df_1["risk_score_v2"] = risk_df_1.apply(
    calculate_risk,
    axis=1
)
risk_df_1.head(20)

,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,dept_high_risk_rate,doctor_high_risk_rate,age_bucket,high_risk_chronic,age_risk_score,chronic_age_interaction,department_risk_rank,patient_visit_burden,visit_type_risk,risk_score_v2
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.189950,0.243728,Senior,1,3,90,1.0,1.098612,0.205917,Medium
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,0.202209,0.156118,Adult,0,2,30,3.0,1.609438,0.200931,Low
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,0.207923,0.174905,Adult,0,2,25,6.0,1.609438,0.205917,Medium
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,0.189950,0.210084,Senior,0,3,0,1.0,2.079442,0.205917,Medium
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,0.198439,0.197628,Senior,1,3,80,2.0,1.791759,0.197159,Medium
5,6,1803,2025-12-20,General,OPD,0.70,High,157,32,M,...,0.198439,0.150579,Adult,0,2,0,2.0,1.386294,0.200931,Low
6,7,4118,2025-11-12,Cardiology,OPD,6.40,Low,121,10,F,...,0.189950,0.239837,Child,0,1,10,1.0,2.197225,0.200931,Low
7,8,1868,2025-09-18,ER,ICU,26.86,High,107,65,M,...,0.206635,0.198444,Senior,0,3,0,5.0,1.791759,0.197159,Medium
8,9,295,2026-01-17,General,ER,8.31,Medium,134,51,M,...,0.198439,0.157258,Middle,0,2,0,2.0,2.079442,0.205917,Low
9,10,4442,2025-12-07,ER,ER,23.22,Low,107,49,F,...,0.206635,0.198444,Middle,0,2,49,5.0,2.302585,0.205917,Medium


In [23]:
#Risk Model Dataset
target = "risk_score_v2"

features = [

   # "age",
   # "gender",
   # "city",
   # "insurance_provider",
    "chronic_flag",
   # "department",
    "visit_type",
   # "doctor_id",
    "visit_count",
    "patient_avg_los",
    "patient_high_risk_rate",
    "dept_high_risk_rate",
   # "doctor_high_risk_rate",
    "age_bucket",
    "high_risk_chronic",
    "age_risk_score",
    "chronic_age_interaction",
   # "department_risk_rank",
    "patient_visit_burden",
    "visit_type_risk"
]

# Ensure risk_df_1 is properly initialized with risk_score_v2 before filtering
# This re-initializes risk_df_1 from df_visit_risk and re-adds risk_score_v2
# Assumes calculate_risk function is already defined (from cell 2oveNo4l4hsq)
risk_df_1 = df_visit_risk.copy()
risk_df_1["risk_score_v2"] = risk_df_1.apply(calculate_risk, axis=1)

risk_df_1 = (
   risk_df_1[features + [target]]
     .dropna()
)
print(risk_df_1.shape)
risk_df_1.head(5)

(25000, 13)


,chronic_flag,visit_type,visit_count,patient_avg_los,patient_high_risk_rate,dept_high_risk_rate,age_bucket,high_risk_chronic,age_risk_score,chronic_age_interaction,patient_visit_burden,visit_type_risk,risk_score_v2
0,1,ER,2,3.725000,0.000000,0.189950,Senior,1,3,90,1.098612,0.205917,Medium
1,1,OPD,4,32.025000,0.500000,0.202209,Adult,0,2,30,1.609438,0.200931,Low
2,1,ER,4,20.542500,0.000000,0.207923,Adult,0,2,25,1.609438,0.205917,Medium
3,0,ER,7,28.165714,0.428571,0.189950,Senior,0,3,0,2.079442,0.205917,Medium
4,1,ICU,5,22.988000,0.200000,0.198439,Senior,1,3,80,1.791759,0.197159,Medium


In [24]:
#preparing your data for Machine Learning by converting
#text/categorical values into numbers,because
#algorithms such as
#Random Forest,
#XGBoost,
#CatBoost,
#Logistic Regression,
#and Decision Trees cannot work directly with text values.

from sklearn.preprocessing import LabelEncoder

# Re-create risk_df_1 and its target column to ensure consistency
# This assumes df_visit_risk and calculate_risk are available in the kernel
risk_df_1 = df_visit_risk.copy()
risk_df_1["risk_score_v2"] = risk_df_1.apply(calculate_risk, axis=1)

# Define target and features again to be consistent with g_qdrlQ9B6TO
#target = "risk_score_v2"
#features = [
    #"age", "gender", "city", "insurance_provider", "chronic_flag",
    #"department", "visit_type", "doctor_id", "visit_count",
    #"patient_avg_los", "patient_high_risk_rate", "dept_high_risk_rate",
    #"doctor_high_risk_rate", "age_bucket", "high_risk_chronic",
    #"age_risk_score", "chronic_age_interaction", "department_risk_rank",
    #"patient_visit_burden", "visit_type_risk"
#]

target = "risk_score_v2"
features = [
    "chronic_flag",
    "visit_type", "visit_count",
    "patient_avg_los", "patient_high_risk_rate", "dept_high_risk_rate",
    "doctor_high_risk_rate", "age_bucket", "high_risk_chronic",
    "age_risk_score", "chronic_age_interaction",
    "patient_visit_burden", "visit_type_risk"
]


# Filter risk_df_1 to selected features and target
risk_df_1 = risk_df_1[features + [target]].dropna()

# Explicitly create a copy to avoid SettingWithCopyWarning
X = risk_df_1[features].copy()

# Ensure y is a 1D Series. .squeeze() handles cases where risk_df_1[target] might return a DataFrame (due to duplicate column names) or a Series.
y = risk_df_1[target].squeeze()

categorical_cols = (
    X.select_dtypes(
        include=[
            "object",
            "category"
        ]
    )
    .columns
)

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    # Explicitly cast to 'int32' after transformation to resolve FutureWarning
    X.loc[:, col] = le.fit_transform(
        X[col].astype(str)
    ).astype('int32')

    encoders[col] = le

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(y)

print(
    dict(
        zip(
            target_encoder.classes_,
            range(
                len(
                    target_encoder.classes_
                )
            )
        )
    )
)


{'High': 0, 'Low': 1, 'Medium': 2}


/tmp/ipykernel_991/57748237.py:65: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[3 0 0 ... 2 1 0]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X.loc[:, col] = le.fit_transform(


In [25]:
# performing Feature Selection using Mutual Information (MI)
# to determine which features are most useful for predicting the target variable.
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X,
    y_encoded
)

mi_df = pd.DataFrame({
    "Feature": X.columns,
    "Score": mi_scores
})

mi_df.sort_values(
    "Score",
    ascending=False
)

,Feature,Score
3,patient_avg_los,0.247181
10,chronic_age_interaction,0.182103
0,chronic_flag,0.118076
2,visit_count,0.073700
8,high_risk_chronic,0.072633
11,patient_visit_burden,0.070218
9,age_risk_score,0.050093
7,age_bucket,0.048388
4,patient_high_risk_rate,0.044265
5,dept_high_risk_rate,0.004346


In [26]:
selected_features = (
    mi_df
      .head(12)
      ["Feature"]
      .tolist()
)

X = X[selected_features]

In [27]:
from sklearn.model_selection import train_test_split

X_train_1, X_test_1, y_train_1, y_test_1 = (
    train_test_split(
        X,
        y_encoded,
        test_size=0.20,
        stratify=y_encoded,
        random_state=42
    )
)

In [28]:
from imblearn.over_sampling import SMOTE

smote_1 = SMOTE(
    random_state=42
)

X_train_smote_1, y_train_smote_1 = (
    smote_1.fit_resample(
        X_train_1,
        y_train_1
    )
)

In [29]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [31]:
#Train usin Randome forest using the parameters Best Random Forest Parameters:
#{'n_estimators': 500, 'min_samples_split': 2, 'max_depth': None}

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf_2 = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_estimators=500,
    min_samples_split=2,
    max_depth=None
)

rf_2.fit(
    X_train_smote_1,
    y_train_smote_1
)

RandomForestClassifier(class_weight='balanced', n_estimators=500,
                       random_state=42)

In [35]:
#Train using Xgboost Algorithm with the parameters Best xgboost Parameters:
#{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.1}

from xgboost import XGBClassifier

# Identify categorical features from the selected_features list that need conversion
# Based on earlier LabelEncoding, 'visit_type' and 'age_bucket' were categorical.
categorical_cols_for_xgb = [col for col in ['visit_type', 'age_bucket'] if col in X_train_smote_1.columns]

# Ensure these columns are of 'category' dtype for XGBoost with enable_categorical=True
# Create a copy to ensure modifications are applied to a standalone DataFrame
X_train_smote_1_processed = X_train_smote_1.copy()

# Safely handle X_test_1, as it might not always exist or be a DataFrame
if 'X_test_1' in locals() and isinstance(X_test_1, pd.DataFrame):
    X_test_1_processed = X_test_1.copy()
else:
    X_test_1_processed = None

for col in categorical_cols_for_xgb:
    # Convert to int first, as SMOTE might introduce floats, then to category
    X_train_smote_1_processed[col] = X_train_smote_1_processed[col].round().astype(int).astype('category')
    # Also apply to X_test_1_processed for consistency if it exists
    if X_test_1_processed is not None and col in X_test_1_processed.columns:
        X_test_1_processed[col] = X_test_1_processed[col].round().astype(int).astype('category')

xgb_2 = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    subsample=0.8,
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    random_state=42,
    enable_categorical=True # This is crucial for handling categorical features in XGBoost
)

xgb_2.fit(
    X_train_smote_1_processed, # Use the processed DataFrame
    y_train_smote_1
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None, num_class=3, ...)

In [37]:
# TRain using CatBoost with the parameters Best Catboost Parameters:
#{'learning_rate': 0.1, 'iterations': 500, 'depth': 8}

from catboost import CatBoostClassifier

# Define categorical features for CatBoost based on the current selected_features
categorical_features_for_catboost = [
    'chronic_flag',
    'visit_type',
    'age_bucket',
    'high_risk_chronic',
    'age_risk_score'
]

cat_2 = CatBoostClassifier(
    verbose=0,
    random_state=42,
    learning_rate=0.1,
    iterations=500,
    depth=8
)

cat_2.fit(
    X_train_smote_1,
    y_train_smote_1,
    cat_features=categorical_features_for_catboost # Pass the list of categorical features here
)

CatBoostClassifier(depth=8, iterations=500, learning_rate=0.1, random_state=42, verbose=0)

In [39]:
#Print F1 score for Cat boost
from sklearn.metrics import (
    accuracy_score,
    f1_score
)
#print f1_score
print(f1_score(y_test_1, cat_2.predict(X_test_1), average='macro'))
#print confusion matrix
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test_1, cat_2.predict(X_test_1)))
#print accuracy, recall percentage
print(f"Accuracy: {accuracy_score(y_test_1, cat_2.predict(X_test_1))}")

0.6275358316252025
[[ 120    0  121]
 [   0 3219  299]
 [ 163  476  602]]
Accuracy: 0.7882


In [41]:
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

models_2 = {
    "Random Forest": rf_2,

    "XGBoost": xgb_2,

    "CatBoost": cat_2
}

results_2 = []

for name, model_2 in models_2.items():
    if name == "XGBoost":
        # For XGBoost, use the X_test_1_processed DataFrame that has 'category' dtypes
        # This DataFrame was prepared in cell wOgltOIYGEFv
        pred_2 = model_2.predict(X_test_1_processed)
    else:
        # For other models, use the original X_test_1 (which has integer-encoded categorical features)
        pred_2 = model_2.predict(X_test_1)

    results_2.append({

        "Model": name,

        "Accuracy":
        accuracy_score(
            y_test_1,
            pred_2
        ),

        "F1 Macro":
        f1_score(
            y_test_1,
            pred_2,
            average="macro"
        )
    })

results_df_2 = pd.DataFrame(results_2)

results_df_2.sort_values(
    "F1 Macro",
    ascending=False
)

,Model,Accuracy,F1 Macro
2,CatBoost,0.7882,0.627536
1,XGBoost,0.7754,0.603854
0,Random Forest,0.7716,0.599521


In [42]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    pred_1
))

              precision    recall  f1-score   support

           0       0.43      0.42      0.42      1007
           1       0.53      0.76      0.63      2494
           2       0.29      0.10      0.15      1499

    accuracy                           0.49      5000
   macro avg       0.42      0.42      0.40      5000
weighted avg       0.44      0.49      0.44      5000



In [43]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_1,
    pred_2
))

              precision    recall  f1-score   support

           0       0.42      0.50      0.46       241
           1       0.87      0.92      0.89      3518
           2       0.59      0.49      0.53      1241

    accuracy                           0.79      5000
   macro avg       0.63      0.63      0.63      5000
weighted avg       0.78      0.79      0.78      5000



In [44]:
#save the best model
import joblib

# Based on the F1 Macro score in results_df_2, best_xgb_2 has the highest score
best_model = cat_2

joblib.dump(
    best_model,
    "best_visit_risk_model.pkl"
)

best_model.save_model("best_visit_risk_model.json")



joblib.dump(
    target_encoder,
    "target_encoder.pkl"
)
# save feature encoder
feature_encoders = {}

for col, le in encoders.items():
    feature_encoders[col] = le

joblib.dump(
    feature_encoders,
    "feature_encoders.pkl"
)

joblib.dump(
    selected_features,
    "selected_features.pkl"
)


['selected_features.pkl']